In [1]:
import datetime
import os
import re
import cdflib
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import re
from time import sleep
import json

下载函数 
获取网址中的所有链接
链接下载，附带报错处理 
错误时不写入文件

In [2]:
def search_url(url,filename):

    response = requests.get(url)
    html_content = response.text

    soup = BeautifulSoup(html_content, 'html.parser')

    file_urls = []
    file_elements = soup.find_all('a', href=True)  # 找到所有带有href属性的<a>元

    for element in file_elements:
        file_url = element['href']
        file_urls.append(file_url)
    urls=[string for string in file_urls if re.match(filename, string)]
    response.close()
    return urls
def requests_downlaod(url,save_path,proxies=None):
    try:
        response = requests.get(url, stream=True,proxies=proxies)
        if response.status_code == 200:
            total_size = int(response.headers.get("content-length", 0))
            block_size = 1024
            progress_bar = tqdm(total=total_size, unit="B", unit_scale=True)
            buffer = bytearray()  # 创建字节缓冲区
            for data in response.iter_content(block_size):
                progress_bar.update(len(data))
                buffer.extend(data)  # 将下载的数据添加到缓冲区
            progress_bar.close()
            
            # 将缓冲区中的数据写入文件
            with open(save_path, "wb") as file:
                file.write(buffer)
            return True
        response.close()
    except requests.exceptions.RequestException as e:
        print(" url error: "+str(e)+',sleep 60s')
        sleep(60)
        return False
def downlaod_model(model):
    url_path_0='https://lasp.colorado.edu/maven/sdc/public/data/sci/'
    with open("data_format.json", "r") as file:
        data = json.load(file)
    save_dir = data["save_path"]
    data_model = data["data_model"]
    model_key = {}
    for key, value in data_model.items():
        model_key[key] = value[:3]

    model_url,filename,pathname=model_key[model]
    url = url_path_0+model_url
    save_path =  save_dir+pathname
    return [url,save_path,filename]
def find_downloaded_file(file_names, element):
    for file_name in file_names:
        if element in file_name:
            return True
    return False
def file_check(file_names,save_path,model):
    if model == 'cdf':
        for f in file_names:
            try :
                data=cdflib.cdfread.CDF(save_path+f)
                print(data)
                print(f,'true')
                continue
            except Exception as e:
                os.remove(os.path.join(save_path,f))
                print('delete:' ,f,e)

测试端口

In [3]:
port=7897
proxies = {
 "http": "http://127.0.0.1:"+str(port),
 "https": "http://127.0.0.1:"+str(port),
}
try:
    response = requests.get("https://www.google.com/",proxies=proxies,timeout=5)
    print(response.status_code)
    if response.status_code == 200:
        print(True)
    response.close()
except requests.exceptions.RequestException as e:
    print(False)
    print(e)
try:
    response = requests.get("https://www.baidu.com/",proxies=proxies,timeout=5)
    print(response.status_code)
    if response.status_code == 200:
        print(True)
    response.close()
except requests.exceptions.RequestException as e:
    print(False)
    print(e)

200
True
200
True


主程序  
获取所有已经存在的文件  

In [4]:
start_date   = datetime.date(2014, 10, 1)
end_date     = datetime.date(2020, 6, 1)

current_date = start_date

models=['SWEA_spec','SWEA_pad','LPW_wave','MAG_ss','MAG_ss1s',"STATIC","KP",'LPW_mrgscpot','SWEA_pad_svy','LPW_lpnt']
model="LPW_burstlf"#'SWEA_pad_svy'#'LPW_wn' #'SWEA_pad_svy'  #'MAG_pc1s'#'MAG'

url_path,save_path,filestyle = downlaod_model(model)
file_names = os.listdir(save_path)
#file_check(file_names,save_path,'cdf')
with open("E:/MAVEN/lists/"+model+'_list.txt', 'w') as file:
    for item in file_names:
        file.write(str(item) + '\n') 

循环所有日期  
跳过已下载文件  

In [5]:
#get all url:
while current_date <= end_date:
    date=str(current_date.strftime("%Y%m%d"))

    if find_downloaded_file(file_names, date):
        print(date,model,'skip')
        start_date=current_date
        current_date+=datetime.timedelta(days=1)
        continue
    year=current_date.year
    month=current_date.month
    yyyymm=str(current_date.strftime("%Y/%m/"))
    urls    = search_url(url_path+yyyymm,filestyle)
    print(len(urls),'urls in ', yyyymm)
    for url in urls:
        filename = str(url)
        date=re.findall(r"\d{8}", filename)
        if find_downloaded_file(file_names, filename):
            print(date,model,'exist')
            continue
        logic = requests_downlaod(url_path+yyyymm+filename,save_path+filename,proxies=proxies)
        if logic == False: 
            os.remove(os.path.join(save_path,filename))
            print('remove',date)
        print('\n',date,model,logic,end='\n')
    month+=1
    if month == 13:
        month=1
        year+=1
    current_date = datetime.date(year, month, 1)

17 urls in  2014/10/
['20141012'] LPW_burstlf exist
['20141013'] LPW_burstlf exist
['20141014'] LPW_burstlf exist
['20141015'] LPW_burstlf exist
['20141016'] LPW_burstlf exist
['20141017'] LPW_burstlf exist
['20141018'] LPW_burstlf exist
['20141019'] LPW_burstlf exist
['20141020'] LPW_burstlf exist
['20141021'] LPW_burstlf exist
['20141022'] LPW_burstlf exist
['20141023'] LPW_burstlf exist
['20141024'] LPW_burstlf exist
['20141025'] LPW_burstlf exist
['20141029'] LPW_burstlf exist
['20141030'] LPW_burstlf exist
['20141031'] LPW_burstlf exist
20141101 LPW_burstlf skip
20141102 LPW_burstlf skip
20141103 LPW_burstlf skip
20141104 LPW_burstlf skip
20 urls in  2014/11/
['20141101'] LPW_burstlf exist
['20141102'] LPW_burstlf exist
['20141103'] LPW_burstlf exist
['20141104'] LPW_burstlf exist


100%|██████████| 86.1M/86.1M [00:12<00:00, 6.67MB/s]



 ['20141105'] LPW_burstlf True


100%|██████████| 91.9M/91.9M [00:11<00:00, 7.88MB/s]



 ['20141106'] LPW_burstlf True


...

100%|██████████| 43.2M/43.2M [00:06<00:00, 7.19MB/s]



 ['20150111'] LPW_burstlf True


100%|██████████| 45.1M/45.1M [00:06<00:00, 7.24MB/s]



 ['20150112'] LPW_burstlf True


  5%|▌         | 1.95M/38.4M [00:10<00:15, 2.36MB/s]

输出list文件记录所有已下载文件

In [ ]:
file_names = os.listdir(save_path)
with open("E:/MAVEN/lists/"+model+'_list.txt', 'w') as file:
    for item in file_names:
        file.write(str(item) + '\n')